<a href="https://colab.research.google.com/github/SUZUB/ATU/blob/main/ATU_GPU.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ==============================================================================
# Cell 1: Infrastructure & Cloud Sync
# ==============================================================================
import os
from google.colab import drive

# 1. Public Git clone / pull
if not os.path.exists('ATU') and not os.path.exists('/content/ATU'):
    !git clone https://github.com/SUZUB/ATU.git
if os.path.exists('/content/ATU'):
    %cd /content/ATU
elif os.path.exists('ATU'):
    %cd ATU
!git pull origin main

# Install project requirements
!pip install -q -r requirements.txt

# 2. Google Drive Mount
drive.mount('/content/drive')

# 3. Persistent Drive Workspace & Subdirectory Creation
DRIVE_WORKSPACE = '/content/drive/MyDrive/ATU_Workspace'
os.makedirs(DRIVE_WORKSPACE, exist_ok=True)

DATASET_PATH = os.path.join(DRIVE_WORKSPACE, 'datasets')
MODEL_CHECKPOINTS = os.path.join(DRIVE_WORKSPACE, 'checkpoints')
RESULTS_PATH = os.path.join(DRIVE_WORKSPACE, 'results')

os.makedirs(DATASET_PATH, exist_ok=True)
os.makedirs(MODEL_CHECKPOINTS, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

print("\nInfrastructure & Cloud Sync complete.")
print(f"Project Directory: {os.getcwd()}")
print(f"Drive Workspace:   {DRIVE_WORKSPACE}")

In [ ]:
# ==============================================================================
# Cell 2: Imports & Setup
# ==============================================================================
# Standard library imports
import os
import sys
import json
import time
from pathlib import Path
from typing import Optional, Tuple, List, Dict, Any

# Third-party Data, ML, and RL Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import gymnasium as gym
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.vec_env import DummyVecEnv

# Ensure ATU root directory is in sys.path
atu_root = Path.cwd()
if str(atu_root) not in sys.path:
    sys.path.insert(0, str(atu_root))

# Custom src module imports
from market_data_processor import MarketDataProcessor
from sentiment_processor import SentimentProcessor
from constrained_trading_env import ConstrainedTradingEnv
from train_and_backtest import create_final_state_vector
from src.data_processing import load_archive_headlines, generate_intraday_5min_market_data
from src.train import calculate_financial_metrics

# Device Auto-detection
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Imports & Setup complete.")
print(f"PyTorch Version:  {torch.__version__}")
print(f"Active Device:    {DEVICE.upper()}")
if torch.cuda.is_available():
    print(f"GPU Hardware:     {torch.cuda.get_device_name(0)}")

In [ ]:
# ==============================================================================
# Cell 3: Data Ingestion & State Vector Merging
# ==============================================================================
# 1. Define dataset directory in persistent Drive Workspace
DRIVE_DATA_DIR = '/content/drive/MyDrive/ATU_Workspace/datasets'

# 2. Load headlines using load_archive_headlines (allow_synthetic_fallback=True, max_rows=200)
print(f"Ingesting headlines from: {DRIVE_DATA_DIR}")
headlines_df = load_archive_headlines(
    archive_dir=DRIVE_DATA_DIR,
    ticker="AAPL",
    max_rows=200,
    allow_synthetic_fallback=True
)
print(f"Headlines Ingested: {len(headlines_df):,} records for ticker 'AAPL'")

# 3. Process FinBERT news sentiment (batch_size=32)
print(f"\nScoring news headlines via FinBERT on device: {DEVICE.upper()} (batch_size=32)...")
sentiment_processor = SentimentProcessor(device=DEVICE)
sentiment_df = sentiment_processor.process_news_dataframe(headlines_df, batch_size=32)
print(f"FinBERT Sentiment Output: {len(sentiment_df):,} trading days scored")

# 4. Generate & process aligned 5-min intraday market data (File 1)
market_start_date = sentiment_df["date"].min() if (sentiment_df is not None and not sentiment_df.empty) else "2024-01-02"
print(f"\nGenerating aligned 5-minute candles starting from anchor date: {market_start_date}...")
market_processor = MarketDataProcessor()
raw_candles = generate_intraday_5min_market_data(n_rows=1500, start_date=market_start_date, initial_price=150.0)
market_df = market_processor.process_dataframe(raw_candles)
print(f"Market Features Engineered: {market_df.shape[0]:,} candles x {market_df.shape[1]} features")

# 5. Create final_state_vector and print its shape/columns
final_state_vector = create_final_state_vector(market_df, sentiment_df)
sentiment_series = final_state_vector["daily_sentiment"]
non_zero_pct = float((sentiment_series != 0.0).mean()) * 100.0

print(f"\nFinal State Vector Created successfully:")
print(f"  Shape:              {final_state_vector.shape}")
print(f"  Columns:            {list(final_state_vector.columns)}")
print(f"  Non-zero Sentiment: {non_zero_pct:.1f}% of state vectors")
display(final_state_vector.head(3))

In [ ]:
# ==============================================================================
# Cell 4: Multi-Agent Ensemble RL (PPO + A2C) & Institutional Backtest
# ==============================================================================
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, A2C
from stable_baselines3.common.vec_env import DummyVecEnv

# ------------------------------------------------------------------------------
# 1. Data Integration (Chronological 80% Train, 20% Test Split)
# ------------------------------------------------------------------------------
assert "final_state_vector" in globals() and final_state_vector is not None, (
    "final_state_vector is missing from memory! Please execute Cell 3 first."
)

split_idx = int(len(final_state_vector) * 0.8)
train_df = final_state_vector.iloc[:split_idx].copy()
test_df = final_state_vector.iloc[split_idx:].copy()

print("Data Integration (No Synthetic Data):")
print(f"  Total Data:     {len(final_state_vector):,} candles")
print(f"  Training Set:   {len(train_df):,} candles ({len(set(pd.to_datetime(train_df.index).date))} days)")
print(f"  Test Set:       {len(test_df):,} candles ({len(set(pd.to_datetime(test_df.index).date))} days)")

# ------------------------------------------------------------------------------
# 2. Vectorize Training Environment
# ------------------------------------------------------------------------------
train_env = DummyVecEnv([lambda: ConstrainedTradingEnv(df=train_df)])

# ------------------------------------------------------------------------------
# 3. Dual Model Setup (PPO + A2C) & Persistent Memory Check
# ------------------------------------------------------------------------------
checkpoint_dir = "/content/drive/MyDrive/ATU_Workspace/checkpoints"
os.makedirs(checkpoint_dir, exist_ok=True)
ppo_checkpoint_path = os.path.join(checkpoint_dir, "ppo_trading_agent_v1.zip")
a2c_checkpoint_path = os.path.join(checkpoint_dir, "a2c_trading_agent_v1.zip")
tensorboard_dir = "/content/drive/MyDrive/ATU_Workspace/training_logs"
os.makedirs(tensorboard_dir, exist_ok=True)

# --- Model 1: PPO Agent ---
if os.path.exists(ppo_checkpoint_path):
    print(f"\n[Persistent Memory] Found existing PPO checkpoint at: {ppo_checkpoint_path}")
    print("Loading PPO model onto CPU and attaching to train environment (preventing amnesia)...")
    ppo_model = PPO.load(ppo_checkpoint_path, env=train_env, device="cpu")
else:
    print(f"\n[Fresh Initialization] No existing PPO checkpoint found at {ppo_checkpoint_path}.")
    print("Instantiating fresh PPO model with device='cpu'...")
    ppo_model = PPO(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=0.0003,
        n_steps=2048,
        batch_size=64,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.2,
        ent_coef=0.01,
        verbose=1,
        seed=42,
        device="cpu",
        tensorboard_log=os.path.join(tensorboard_dir, "ppo"),
    )

# --- Model 2: A2C Agent ---
if os.path.exists(a2c_checkpoint_path):
    print(f"\n[Persistent Memory] Found existing A2C checkpoint at: {a2c_checkpoint_path}")
    print("Loading A2C model onto CPU and attaching to train environment (preventing amnesia)...")
    a2c_model = A2C.load(a2c_checkpoint_path, env=train_env, device="cpu")
else:
    print(f"\n[Fresh Initialization] No existing A2C checkpoint found at {a2c_checkpoint_path}.")
    print("Instantiating fresh A2C model with device='cpu'...")
    a2c_model = A2C(
        policy="MlpPolicy",
        env=train_env,
        learning_rate=0.0007,
        n_steps=5,
        gamma=0.99,
        gae_lambda=1.0,
        ent_coef=0.01,
        verbose=1,
        seed=42,
        device="cpu",
        tensorboard_log=os.path.join(tensorboard_dir, "a2c"),
    )

# ------------------------------------------------------------------------------
# 4. Dual Model Training (500,000 Timesteps Each, No Amnesia)
# ------------------------------------------------------------------------------
print(f"\n[Training 1/2] Launching deep PPO training for 500,000 timesteps (reset_num_timesteps=False)...")
ppo_model.learn(total_timesteps=500000, reset_num_timesteps=False)
ppo_model.save(os.path.splitext(ppo_checkpoint_path)[0])
print(f"PPO model successfully saved to Drive checkpoint: {ppo_checkpoint_path}")

print(f"\n[Training 2/2] Launching deep A2C training for 500,000 timesteps (reset_num_timesteps=False)...")
a2c_model.learn(total_timesteps=500000, reset_num_timesteps=False)
a2c_model.save(os.path.splitext(a2c_checkpoint_path)[0])
print(f"A2C model successfully saved to Drive checkpoint: {a2c_checkpoint_path}")

# ------------------------------------------------------------------------------
# 5. Walk-Forward Consensus Backtest
# ------------------------------------------------------------------------------
print(f"\n[Backtest] Starting continuous ensemble walk-forward loop across {len(test_df) - 2:,} candles...")
test_env = ConstrainedTradingEnv(df=test_df)
state, info = test_env.reset(seed=42)

portfolio_values = [float(test_env.initial_balance)]
step_returns = []
total_trades_executed = 0
profitable_trades = 0
last_direction = 0
direction_changes = 0

for _ in range(len(test_df) - 2):
    # Predict actions from both agents
    action_ppo, _ = ppo_model.predict(state, deterministic=True)
    action_a2c, _ = a2c_model.predict(state, deterministic=True)

    # Consensus Voting:
    # If action_ppo[0] > 0.1 AND action_a2c[0] > 0.1 -> average of both (Long)
    # If action_ppo[0] < -0.1 AND action_a2c[0] < -0.1 -> average of both (Short)
    # Otherwise -> Hold [0.0]
    if action_ppo[0] > 0.1 and action_a2c[0] > 0.1:
        final_action = np.array([(action_ppo[0] + action_a2c[0]) / 2.0], dtype=np.float32)
    elif action_ppo[0] < -0.1 and action_a2c[0] < -0.1:
        final_action = np.array([(action_ppo[0] + action_a2c[0]) / 2.0], dtype=np.float32)
    else:
        final_action = np.array([0.0], dtype=np.float32)

    # Step the environment with consensus decision
    state, reward, terminated, truncated, step_info = test_env.step(final_action)

    # Append balance and step profit
    current_balance = float(step_info.get("balance", step_info.get("account_balance", portfolio_values[-1])))
    step_profit = float(step_info.get("step_profit", 0.0))
    portfolio_values.append(current_balance)
    step_returns.append(step_profit)

    # Track trade execution accurately
    env_trades = int(step_info.get("total_trades", step_info.get("total_trades_count", 0)))
    total_trades_executed = max(total_trades_executed, env_trades)

    if step_info.get("trade_closed_profit", False):
        profitable_trades += 1

    current_direction = step_info.get("position_direction", 0)
    if current_direction != 0 and current_direction != last_direction:
        direction_changes += 1
    last_direction = current_direction

    # Seamlessly handle 15:15 auto-square-off truncation without breaking backtest loop
    if truncated and not terminated:
        saved_balance = current_balance
        state, info = test_env.reset()
        if hasattr(test_env, "account_balance"):
            test_env.account_balance = saved_balance

total_trades_executed = max(total_trades_executed, direction_changes)
total_fees_paid = float(step_info.get("total_fees_paid", 0.0))
if total_trades_executed == 0 and total_fees_paid > 0.0:
    total_trades_executed = max(1, int(np.round(total_fees_paid / (test_env.initial_balance * 0.0003 * 2))))

print(f"Backtest completed: {len(portfolio_values):,} steps recorded, {total_trades_executed} total trades executed.")

# ------------------------------------------------------------------------------
# 6. Compute & Print Institutional Performance Metrics Report
# ------------------------------------------------------------------------------
metrics = calculate_financial_metrics(
    portfolio_values=portfolio_values,
    step_returns=step_returns,
    total_trades=total_trades_executed,
    profitable_trades=profitable_trades,
    initial_balance=float(test_env.initial_balance),
    final_balance=float(portfolio_values[-1]),
    total_fees_paid=total_fees_paid,
)

# Compute Profit Factor (Gross Profits / abs(Gross Losses)) safely to prevent KeyError
pos_profits = sum(r for r in step_returns if r > 0.0)
neg_losses = abs(sum(r for r in step_returns if r < 0.0))
profit_factor = float(pos_profits / neg_losses) if neg_losses > 1e-9 else (float("inf") if pos_profits > 0.0 else 0.0)
metrics["profit_factor"] = profit_factor

# Persist backtest results and metrics to Google Drive RESULTS_PATH
RESULTS_PATH = "/content/drive/MyDrive/ATU_Workspace/results"
os.makedirs(RESULTS_PATH, exist_ok=True)

backtest_df = pd.DataFrame({
    "step": list(range(len(portfolio_values))),
    "portfolio_value": portfolio_values,
    "step_return": [0.0] + step_returns[:len(portfolio_values) - 1],
})
backtest_csv_path = os.path.join(RESULTS_PATH, "backtest_portfolio_history.csv")
backtest_df.to_csv(backtest_csv_path, index=False)

metrics_json_path = os.path.join(RESULTS_PATH, "evaluation_metrics.json")
with open(metrics_json_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)

print("\n" + "=" * 65)
print("      ENSEMBLE (PPO + A2C) INSTITUTIONAL PERFORMANCE REPORT    ")
print("=" * 65)
print(f" Initial Balance:          ₹{metrics['initial_balance']:,.2f}")
print(f" Final Compounded Balance:  ₹{metrics['final_balance']:,.2f}")
print(f" Net Profit (After Fees):  ₹{metrics['net_profit']:+,.2f} ({metrics['net_return_pct']:+.2f}%)")
print(f" Gross Profit (Pre-Fees):  ₹{metrics['gross_profit']:+,.2f} ({metrics['gross_return_pct']:+.2f}%)")
print(f" Total Brokerage & STT:    ₹{metrics['total_fees_paid']:,.2f}")
print(f" Total Trades Executed:    {metrics['total_trades']:,} (Win Rate: {metrics['win_rate_pct']:.1f}%)")
print(f" Maximum Drawdown:         {metrics['max_drawdown_pct']:.2f}%")
print(f" Sharpe Ratio (Ann.):      {metrics['sharpe_ratio']:.4f}")
print(f" Sortino Ratio (Ann.):     {metrics['sortino_ratio']:.4f}")
profit_factor_val = metrics.get("profit_factor", profit_factor)
if profit_factor_val != float("inf") and not np.isnan(profit_factor_val):
    print(f" Profit Factor:            {profit_factor_val:.2f}")
else:
    print(f" Profit Factor:            N/A (No Losses)")
print("=" * 65)
print(f"Results CSV: {backtest_csv_path}")
print(f"Metrics:     {metrics_json_path}")

# ------------------------------------------------------------------------------
# 7. Equity Curve Visualization
# ------------------------------------------------------------------------------
plt.figure(figsize=(12, 5))
plt.plot(portfolio_values, color="#2563EB", linewidth=1.8, label="Ensemble Portfolio Equity (₹)")
plt.axhline(test_env.initial_balance, color="gray", linestyle="--", alpha=0.7, label="Initial Capital")
plt.title("ATU Multi-Agent Ensemble Out-of-Sample Equity Curve (AAPL)", fontsize=13)
plt.xlabel("Intraday 5-Min Steps", fontsize=11)
plt.ylabel("Portfolio Balance (₹)", fontsize=11)
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
equity_curve_path = os.path.join(RESULTS_PATH, "equity_curve.png")
plt.savefig(equity_curve_path, dpi=150)
plt.show()
print(f"Saved equity curve plot to: {equity_curve_path}")
